# E3: Data Split

**Spec sections**: 3.2 (OOT lock), 5.1, 5.3, 5.4, 5.5, Appendix D.1–D.2

Assign windows, create R-folds / T-folds / back-test splits, and produce the split report.

In [0]:
import sys, os, shutil, tempfile

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import common

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
ARTIFACTS = common.artifacts_path(cfg)
REPORTS = os.path.join(ARTIFACTS, "reports")
FOLDS   = os.path.join(ARTIFACTS, "folds")
os.makedirs(REPORTS, exist_ok=True)
os.makedirs(FOLDS, exist_ok=True)

id_col     = cfg["columns"]["id"]
date_col   = cfg["columns"]["date"]
target_col = cfg["columns"]["target"]
cat        = cfg["databricks"]["catalog"]
sch        = cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"

_TMP = tempfile.mkdtemp(prefix="e3_")
def save_excel(df, fn):
    tmp = os.path.join(_TMP, fn); df.to_excel(tmp, index=False)
    shutil.copy2(tmp, os.path.join(REPORTS, fn))
    print(f"Saved: {fn}")

print(f"Config hash: {common.config_hash(cfg)[:12]}")
print(f"Master table: {MASTER_TBL}")
print(f"Folds dir: {FOLDS}")

In [0]:
# Read master table from E2-B output
master = spark.table(MASTER_TBL).toPandas()
row_count_before = len(master)
print(f"Master rows: {row_count_before:,}  |  Columns: {len(master.columns)}")

# Assign windows using common.assign_window
# First drop any existing window/app_month/app_quarter to avoid conflicts
for c in ["window", "app_month", "app_quarter"]:
    if c in master.columns:
        master.drop(columns=[c], inplace=True)

master = common.assign_window(master, cfg)

# Tag rows outside all windows as OUT_OF_SCOPE
master["window"] = master["window"].fillna("OUT_OF_SCOPE")
n_oos = (master["window"] == "OUT_OF_SCOPE").sum()
if n_oos > 0:
    oos = master[master["window"] == "OUT_OF_SCOPE"]
    print(f"\nWARNING: {n_oos} rows outside all windows (OUT_OF_SCOPE)")
    print(f"  Date range: {pd.to_datetime(oos[date_col]).min()} to {pd.to_datetime(oos[date_col]).max()}")

# Assign oot_half from config.oot_report_splits
master["oot_half"] = pd.NA
if "oot_report_splits" in cfg:
    d = pd.to_datetime(master[date_col])
    for half_name, (s, e) in cfg["oot_report_splits"].items():
        mask = (d >= s) & (d < e) & (master["window"] == "OOT")
        master.loc[mask, "oot_half"] = half_name

print(f"\nWindow distribution:")
for w, c in master["window"].value_counts(dropna=False).sort_index().items():
    print(f"  {str(w):15s}  {c:>6,}")
print(f"\nOOT half distribution:")
print(master.loc[master["window"]=="OOT", "oot_half"].value_counts(dropna=False).to_string())

assert len(master) == row_count_before, "Row count changed!"
print(f"\nRow count preserved: {len(master):,}")

In [0]:
# Write back: convert Period columns to strings for Delta compatibility
master_out = master.copy()
for c in ["app_month", "app_quarter"]:
    if c in master_out.columns:
        master_out[c] = master_out[c].astype(str)

sdf = spark.createDataFrame(master_out)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(MASTER_TBL)
print(f"Master table updated: {MASTER_TBL}")
print(f"Rows: {spark.table(MASTER_TBL).count():,}")

In [0]:
# DEV GOOD+BAD only
dev = master[
    (master["window"] == "DEV")
    & (master["sample_status"].isin(cfg["labelled_statuses"]))
].copy()
common.assert_labelled(dev, cfg)
print(f"DEV labelled: {len(dev):,} rows, {(dev[target_col]==1).sum()} Bads")

# Generate R-folds
folds = common.r_folds(dev, cfg)
print(f"R-fold splits: {len(folds)} (n_splits={cfg['cv']['r_folds']['n_splits']} x n_repeats={cfg['cv']['r_folds']['n_repeats']})")

# Build fold DataFrame: app_id, scheme, repeat, fold, part
n_splits = cfg["cv"]["r_folds"]["n_splits"]
rows = []
for i, (train_idx, valid_idx) in enumerate(folds):
    repeat_num = i // n_splits
    fold_num   = i % n_splits
    for idx in train_idx:
        rows.append({"app_id": dev.iloc[idx][id_col], "scheme": "R",
                     "repeat": repeat_num, "fold": fold_num, "part": "train"})
    for idx in valid_idx:
        rows.append({"app_id": dev.iloc[idx][id_col], "scheme": "R",
                     "repeat": repeat_num, "fold": fold_num, "part": "valid"})

rf_df = pd.DataFrame(rows)
rf_path = os.path.join(FOLDS, "r_folds.parquet")
rf_df.to_parquet(rf_path, index=False)
print(f"Saved: {rf_path}  ({len(rf_df):,} rows)")

In [0]:
# T-folds T1-T3
t_splits = common.t_folds(dev, cfg)
print(f"T-fold splits: {len(t_splits)}")

tf_rows = []
for i, (train_idx, valid_idx) in enumerate(t_splits):
    fold_name = list(cfg["cv"]["t_folds"].keys())[i]
    for idx in train_idx:
        tf_rows.append({"app_id": dev.iloc[idx][id_col], "scheme": "T",
                        "repeat": 0, "fold": fold_name, "part": "train"})
    for idx in valid_idx:
        tf_rows.append({"app_id": dev.iloc[idx][id_col], "scheme": "T",
                        "repeat": 0, "fold": fold_name, "part": "valid"})

tf_df = pd.DataFrame(tf_rows)
tf_path = os.path.join(FOLDS, "t_folds.parquet")
tf_df.to_parquet(tf_path, index=False)
print(f"Saved: {tf_path}  ({len(tf_df):,} rows)")

# Report T-fold sizes
for fn in cfg["cv"]["t_folds"]:
    tf = tf_df[tf_df["fold"] == fn]
    for p in ["train", "valid"]:
        n = (tf["part"] == p).sum()
        print(f"  {fn} {p}: {n:,}")

In [0]:
# Backtest split from config.backtest
bt = cfg["backtest"]
d = pd.to_datetime(dev[date_col])
bt_train_mask = (d >= bt["train"][0]) & (d < bt["train"][1])
bt_test_mask  = (d >= bt["test"][0])  & (d < bt["test"][1])

bt_rows = []
for idx in np.where(bt_train_mask)[0]:
    bt_rows.append({"app_id": dev.iloc[idx][id_col], "scheme": "BT",
                    "repeat": 0, "fold": "backtest", "part": "train"})
for idx in np.where(bt_test_mask)[0]:
    bt_rows.append({"app_id": dev.iloc[idx][id_col], "scheme": "BT",
                    "repeat": 0, "fold": "backtest", "part": "valid"})

bt_df = pd.DataFrame(bt_rows)
bt_path = os.path.join(FOLDS, "backtest.parquet")
bt_df.to_parquet(bt_path, index=False)
print(f"Saved: {bt_path}  ({len(bt_df):,} rows)")
print(f"  Train: {(bt_df['part']=='train').sum():,}  |  Valid: {(bt_df['part']=='valid').sum():,}")

In [0]:
import openpyxl

# Sheet 1: Counts and Bads by window (GOOD+BAD)
lab = master[master["sample_status"].isin(cfg["labelled_statuses"])]
w_summary = []
for w in ["DEV", "OOT", "FWD", "STRESS", "OUT_OF_SCOPE"]:
    wdf = lab[lab["window"] == w]
    w_summary.append({"window": w, "records": len(wdf),
                      "bads": int((wdf[target_col]==1).sum()),
                      "bad_rate": (wdf[target_col]==1).mean() if len(wdf) else None})
sheet1 = pd.DataFrame(w_summary)

# Sheet 2: Window x sample_status
sheet2 = master.groupby(["window", "sample_status"]).size().unstack(fill_value=0)

# Sheet 3: DEV_EARLY / DEV_LATE
dev_lab = lab[lab["window"] == "DEV"]
d_dev = pd.to_datetime(dev_lab[date_col])
dev_early_end = cfg["backtest"]["train"][1]
sheet3_rows = []
for seg, mask in [("DEV_EARLY", d_dev < dev_early_end), ("DEV_LATE", d_dev >= dev_early_end)]:
    s = dev_lab[mask]
    sheet3_rows.append({"segment": seg, "records": len(s), "bads": int((s[target_col]==1).sum()),
                        "bad_rate": (s[target_col]==1).mean() if len(s) else None})
sheet3 = pd.DataFrame(sheet3_rows)

# Sheet 4: R-fold valid stats
dev_target = dev.set_index(id_col)[target_col]
dev_br = (dev[target_col]==1).mean()
rf_valid = rf_df[rf_df["part"]=="valid"].copy()
rf_valid["target"] = rf_valid["app_id"].map(dev_target)
rf_stats = []
for (rep, fld), grp in rf_valid.groupby(["repeat", "fold"]):
    n = len(grp); bads = int(grp["target"].sum()); br = bads/n if n else 0
    qmix = grp.merge(dev[[id_col, "app_quarter"]], left_on="app_id", right_on=id_col, how="left")["app_quarter"].value_counts(normalize=True).to_dict()
    rf_stats.append({"repeat": rep, "fold": fld, "n": n, "bads": bads,
                     "bad_rate": br, "br_diff_pp": (br - dev_br)*100,
                     "quarter_mix": str(qmix)})
sheet4 = pd.DataFrame(rf_stats)

# Sheet 5: T-fold stats
tf_stats = []
for fn in cfg["cv"]["t_folds"]:
    for p in ["train", "valid"]:
        sub = tf_df[(tf_df["fold"]==fn) & (tf_df["part"]==p)]
        ids = sub["app_id"]
        targets = ids.map(dev_target)
        tf_stats.append({"fold": fn, "part": p, "n": len(sub), "bads": int(targets.sum())})
sheet5 = pd.DataFrame(tf_stats)

# Write to Excel
xl_path = os.path.join(_TMP, "02_split.xlsx")
with pd.ExcelWriter(xl_path) as writer:
    sheet1.to_excel(writer, sheet_name="window_summary", index=False)
    sheet2.to_excel(writer, sheet_name="window_x_status")
    sheet3.to_excel(writer, sheet_name="dev_early_late", index=False)
    sheet4.to_excel(writer, sheet_name="r_fold_stats", index=False)
    sheet5.to_excel(writer, sheet_name="t_fold_stats", index=False)
shutil.copy2(xl_path, os.path.join(REPORTS, "02_split.xlsx"))
print("Saved: 02_split.xlsx")
display(sheet1)
display(sheet3)

In [0]:
print("=" * 60)
print("ACCEPTANCE CRITERIA (E3)")
print("=" * 60)
results = []

# 1. Row count unchanged, no null window
mt_count = spark.table(MASTER_TBL).count()
results.append(("Master row count unchanged", mt_count == row_count_before))
results.append(("No null window", master["window"].notna().all()))

# 2. GOOD+BAD counts match expected_counts for ALL, DEV, DEV_EARLY, DEV_LATE
expected = cfg["expected_counts"]
lab_all = master[(master["window"].isin(["DEV","OOT"])) & (master["sample_status"].isin(cfg["labelled_statuses"]))]
dev_lab = lab_all[lab_all["window"]=="DEV"]
d_dev = pd.to_datetime(dev_lab[date_col])

for seg, df_seg in [
    ("ALL", lab_all),
    ("DEV", dev_lab),
    ("DEV_EARLY", dev_lab[d_dev < cfg["backtest"]["train"][1]]),
    ("DEV_LATE", dev_lab[d_dev >= cfg["backtest"]["train"][1]]),
]:
    er, eb = expected[seg]["records"], expected[seg]["bads"]
    ar, ab = len(df_seg), int((df_seg[target_col]==1).sum())
    ok = (er == ar) and (eb == ab)
    results.append((f"{seg} counts ({ar}/{ab} vs {er}/{eb})", ok))

# 3. OOT: one aggregate SQL count only (no OOT rows in pandas)
oot_counts = spark.sql(f"""
    SELECT COUNT(*) AS n, SUM(CASE WHEN target=1 THEN 1 ELSE 0 END) AS bads
    FROM {MASTER_TBL}
    WHERE window='OOT' AND sample_status IN ('GOOD','BAD')
""").toPandas()
oot_n, oot_b = int(oot_counts["n"].iloc[0]), int(oot_counts["bads"].iloc[0])
results.append((f"OOT counts via SQL ({oot_n}/{oot_b} vs {expected['OOT']['records']}/{expected['OOT']['bads']})",
    oot_n == expected["OOT"]["records"] and oot_b == expected["OOT"]["bads"]))

# 4. OOT read lock: load_window must raise
try:
    common.load_window(master, "OOT", cfg)
    results.append(("OOT read lock raises PermissionError", False))
except PermissionError:
    results.append(("OOT read lock raises PermissionError", True))
except Exception as e:
    results.append((f"OOT lock raised {type(e).__name__} (expected PermissionError)", False))

# 5. R-folds: 15 splits, each app_id in exactly one valid fold per repeat, Bad Rate within ±1pp
n_rfolds = len(folds)
results.append((f"15 R-fold splits (got {n_rfolds})", n_rfolds == 15))

rf_valid_check = rf_df[rf_df["part"]=="valid"]
all_unique_per_repeat = True
for rep in rf_valid_check["repeat"].unique():
    rep_valid = rf_valid_check[rf_valid_check["repeat"]==rep]
    if rep_valid["app_id"].duplicated().any():
        all_unique_per_repeat = False
        break
results.append(("Each app_id in exactly one valid fold per repeat", all_unique_per_repeat))

br_ok = all(abs(r["br_diff_pp"]) <= 1.0 for r in rf_stats)
results.append(("R-fold Bad Rates within ±1pp of DEV", br_ok))

# 6. T-fold disjoint, match config dates, all DEV GOOD+BAD
dev_ids = set(dev[id_col])
all_t_disjoint = True
all_t_dev = True
for fn in cfg["cv"]["t_folds"]:
    tr_ids = set(tf_df[(tf_df["fold"]==fn)&(tf_df["part"]=="train")]["app_id"])
    va_ids = set(tf_df[(tf_df["fold"]==fn)&(tf_df["part"]=="valid")]["app_id"])
    if tr_ids & va_ids:
        all_t_disjoint = False
    if not (tr_ids | va_ids) <= dev_ids:
        all_t_dev = False
results.append(("T-fold train/valid disjoint", all_t_disjoint))
results.append(("All T-fold app_ids are DEV GOOD+BAD", all_t_dev))

for desc, ok in results:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {desc}")

n_pass = sum(1 for _,p in results if p)
n_fail = sum(1 for _,p in results if not p)
print(f"\n{n_pass} PASS, {n_fail} FAIL out of {len(results)} checks.")
if n_fail == 0:
    print("\nE3 DATA SPLIT: ALL ACCEPTANCE CRITERIA PASSED")

shutil.rmtree(_TMP, ignore_errors=True)